In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece rouge-score sentence-transformers
from huggingface_hub import notebook_login
notebook_login() # Enter your Hugging Face token here

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00


In [ ]:
import gc
import math
import warnings
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoConfig # Import AutoConfig
)

warnings.filterwarnings("ignore")

# ── Paper hyper-parameters ────────────────────────────────────────────────────
K_GENERATIONS     = 10       # number of responses per question
TEMPERATURE       = 0.5
TOP_P             = 0.99
TOP_K             = 5
ALPHA             = 1e-3     # covariance regularisation
CLIP_PERCENTILE   = 0.2      # p = 0.2 % (three-sigma rule)
MIDDLE_LAYER      = 16       # layer index for sentence embedding (L/2 for 7B)
PENULTIMATE_LAYER = 30       # layer where FC hook is attached (32 layers total)
MAX_NEW_TOKENS    = 64       # cap generation length

# ── Detection thresholds (from paper Appendix H) ─────────────────────────────
THRESHOLDS = {
    "perplexity":         0.535,   # score < threshold  → non-hallucination
    "ln_entropy":         0.153,   # score < threshold  → non-hallucination
    "lexical_similarity": 0.489,   # score > threshold  → non-hallucination
    "eigen_score":       -1.74,    # score < threshold  → non-hallucination
}

# ─────────────────────────────────────────────────────────────────────────────
# 1.  MODEL LOADING  (4-bit quantised for T4 16 GB)
# ─────────────────────────────────────────────────────────────────────────────

def load_model_and_tokenizer(
    model_id: str = "microsoft/phi-2",
) -> Tuple[AutoModelForCausalLM, AutoTokenizer]:
    """
    Load the model in 4-bit NF4 quantisation using bitsandbytes.
    output_hidden_states is set to True globally so every forward pass
    returns all layer hidden states without extra overhead.

    Handles missing pad_token / pad_token_id gracefully for models like
    Phi-2, Mistral, Falcon that don't define one in their config.
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

    # ── Fix 1: ensure tokenizer has a pad token ───────────────────────────
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"   # for batch generation

    # Load model config first to modify pad_token_id before model creation
    config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    if not hasattr(config, "pad_token_id") or config.pad_token_id is None:
        config.pad_token_id = tokenizer.pad_token_id

    # Set output_hidden_states in the config, not as a direct argument to from_pretrained
    config.output_hidden_states = True

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        # output_hidden_states=True,     # Moved to config
        torch_dtype=torch.float16,
        trust_remote_code=True,        # needed for Phi-2
        config=config,                 # Pass the modified config
    )

    model.eval()

    total = sum(p.numel() for p in model.parameters()) / 1e9
    print(f"[✓] Loaded '{model_id}'  |  ~{total:.1f}B parameters  |  4-bit NF4")
    print(f"    pad_token_id = {model.config.pad_token_id}  |  "
          f"eos_token_id = {model.config.eos_token_id}")
    return model, tokenizer


# ─────────────────────────────────────────────────────────────────────────────
# 2.  FEATURE CLIPPING  (Test-Time Hook)
# ─────────────────────────────────────────────────────────────────────────────

class FeatureClipHook:
    """
    Attaches a forward hook to `penultimate_layer` of the model.

    On the first CLIP_PERCENTILE_WARMUP tokens the hook only *observes*
    activations to build a running memory bank; from that point on it
    also clamps extreme values to the [p-th, (100-p)-th] percentile range.

    Usage (context manager is recommended):
        with FeatureClipHook(model, layer=30, p=0.2) as fc:
            outputs = model(**inputs)
    """

    def __init__(
        self,
        model: AutoModelForCausalLM,
        layer: int = PENULTIMATE_LAYER,
        p: float = CLIP_PERCENTILE,
        memory_size: int = 3000,
    ):
        self.layer       = layer
        self.p           = p
        self.memory_bank: List[torch.Tensor] = []
        self.memory_size = memory_size
        self._handle     = None
        self._active     = False

        # Grab the target sub-module (works for LLaMA / Llama-2 architecture)
        target_module = model.model.layers[layer]
        self._handle  = target_module.register_forward_hook(self._hook_fn)

    # ── hook function ──────────────────────────────────────────────────────
    def _hook_fn(
        self,
        module,
        input_tensors,
        output_tensors,
    ):
        # output_tensors may be a tuple; hidden state is the first element
        if isinstance(output_tensors, tuple):
            hidden = output_tensors[0]
        else:
            hidden = output_tensors

        # Flatten spatial dims to collect activations into memory bank
        flat = hidden.detach().reshape(-1, hidden.shape[-1])  # (B*T, d)

        # Update memory bank (FIFO, capped at memory_size)
        self.memory_bank.append(flat.cpu().float())
        total = sum(t.shape[0] for t in self.memory_bank)
        while total > self.memory_size and len(self.memory_bank) > 1:
            total -= self.memory_bank.pop(0).shape[0]

        if not self._active:
            return output_tensors  # warm-up phase: observe only

        # Compute per-neuron clip bounds from memory bank
        bank = torch.cat(self.memory_bank, dim=0)  # (N, d)
        lo   = torch.quantile(bank, self.p / 100.0,         dim=0).to(hidden.device, hidden.dtype)
        hi   = torch.quantile(bank, 1.0 - self.p / 100.0,  dim=0).to(hidden.device, hidden.dtype)

        clipped = hidden.clamp(min=lo, max=hi)

        if isinstance(output_tensors, tuple):
            return (clipped,) + output_tensors[1:]
        return clipped

    # ── context-manager helpers ────────────────────────────────────────────
    def activate(self):
        """Enable clipping (after warm-up is done)."""
        self._active = True

    def deactivate(self):
        self._active = False

    def remove(self):
        if self._handle is not None:
            self._handle.remove()
            self._handle = None

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.remove()


@contextmanager
def feature_clip_context(model, layer=PENULTIMATE_LAYER, p=CLIP_PERCENTILE):
    """Convenience context manager that activates clipping immediately."""
    hook = FeatureClipHook(model, layer=layer, p=p)
    hook.activate()
    try:
        yield hook
    finally:
        hook.remove()


# ─────────────────────────────────────────────────────────────────────────────
# 3.  INTERNAL STATE EXTRACTION  (K generations + sentence embeddings)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class GenerationResult:
    responses:          List[str]           # decoded text responses
    sentence_embeddings: torch.Tensor       # (K, d) — middle-layer last-token
    token_log_probs:    List[List[float]]   # per-token log-probs for each response
    perplexity:         float
    ln_entropy:         float


def _extract_sentence_embedding(
    hidden_states: Tuple[torch.Tensor, ...],
    middle_layer: int = MIDDLE_LAYER,
) -> torch.Tensor:
    """
    Extract the sentence embedding as the hidden state of the *last generated
    token* at the middle layer (layer index = middle_layer).

    hidden_states layout from HF:
        Tuple[ layer_0_hidden, layer_1_hidden, ..., layer_L_hidden ]
        each tensor: (batch=1, seq_len, d)
    We want the hidden state at the *last generated step*
    """
    h = hidden_states[middle_layer]   # (1, T, d)
    return h[0, -1, :].float()        # (d,)


def generate_responses(
    model:        AutoModelForCausalLM,
    tokenizer:    AutoTokenizer,
    question:     str,
    k:            int   = K_GENERATIONS,
    temperature:  float = TEMPERATURE,
    top_p:        float = TOP_P,
    top_k:        int   = TOP_K,
    max_new_tokens: int = MAX_NEW_TOKENS,
    use_feature_clip: bool = False,
    clip_layer:   int   = PENULTIMATE_LAYER,
    clip_p:       float = CLIP_PERCENTILE,
) -> GenerationResult:
    """
    Generate K responses for `question` and collect:
      • decoded text
      • sentence embeddings (middle layer, last token)
      • per-token log-probabilities (for perplexity / LN-entropy)
    """
    # Auto-detect prompt format from model config name
    # Phi-2 uses "Instruct/Output", base models use plain Q&A style
    model_name = getattr(model.config, "_name_or_path", "").lower()
    if "llama-2" in model_name and "chat" in model_name:
        prompt = f"[INST] {question.strip()} [/INST]"
    elif "phi" in model_name:
        prompt = f"Instruct: {question.strip()}\nOutput:"
    else:
        prompt = f"Question: {question.strip()}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    all_responses:    List[str]         = []
    all_embeddings:   List[torch.Tensor] = []
    all_token_lp:     List[List[float]] = []

    hook_ctx = (
        feature_clip_context(model, layer=clip_layer, p=clip_p)
        if use_feature_clip
        else _null_context()
    )

    with hook_ctx:
        for _ in range(k):
            with torch.inference_mode():
                out = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=temperature,
                    top_p=top_p,
                    top_k=top_k,
                    output_hidden_states=True,
                    output_scores=True,
                    return_dict_in_generate=True,
                    pad_token_id=tokenizer.eos_token_id,
                )

            # ── Decode response ──────────────────────────────────────────
            gen_ids  = out.sequences[0, prompt_len:]
            response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
            all_responses.append(response)

            # ── Sentence embedding: last token of middle layer ────────────
            # out.hidden_states is a tuple of length = num_new_tokens
            # Each element: tuple of layer tensors  (batch, cur_len, d)
            # We want the hidden state at the *last generated step*
            last_step_hidden = out.hidden_states[-1]   # last generation step
            emb = _extract_sentence_embedding(last_step_hidden, MIDDLE_LAYER)
            all_embeddings.append(emb)

            # ── Per-token log-probs (for perplexity / LN-entropy) ─────────
            step_log_probs = []
            for score_t, tok_id in zip(out.scores, gen_ids):
                # score_t: (1, vocab_size) logits before sampling
                lp = F.log_softmax(score_t[0], dim=-1)[tok_id].item()
                step_log_probs.append(lp)
            all_token_lp.append(step_log_probs)

            # Free GPU memory between generations
            del out
            torch.cuda.empty_cache()

    # ── Perplexity  (Eq.1 in paper) ───────────────────────────────────────
    all_lp_flat  = [lp for seq in all_token_lp for lp in seq]
    perplexity   = math.exp(-np.mean(all_lp_flat)) if all_lp_flat else float("inf")

    # ── Length-Normalised Entropy  (Eq.2) ────────────────────────────────
    # H = -E_y[ (1/T_y) * Σ_t log p(y_t) ]
    per_seq_avg = [
        -np.mean(seq) for seq in all_token_lp if seq
    ]
    ln_entropy = float(np.mean(per_seq_avg)) if per_seq_avg else float("inf")

    embeddings_tensor = torch.stack(all_embeddings, dim=0)  # (K, d)

    return GenerationResult(
        responses=all_responses,
        sentence_embeddings=embeddings_tensor,
        token_log_probs=all_token_lp,
        perplexity=perplexity,
        ln_entropy=ln_entropy,
    )


@contextmanager
def _null_context():
    """No-op context manager used when feature clipping is disabled."""
    yield


# ─────────────────────────────────────────────────────────────────────────────
# 4.  EIGENSCORE  (Section 3.1)
# ─────────────────────────────────────────────────────────────────────────────

def compute_eigen_score(
    embeddings: torch.Tensor,    # (K, d)
    alpha: float = ALPHA,
) -> Tuple[float, np.ndarray]:
    """
    Compute EigenScore as defined in Eq. 5–6 of the paper.

        Σ = Z^T · J_d · Z          (Z is d×K, J_d is the centering matrix)
        E = (1/K) Σ_i log(λ_i)    (λ_i = eigenvalues of Σ + α·I)

    Steps:
      1. L2-normalise each embedding  (prevents eigenvalue explosion on 4-bit)
      2. Build Z = embeddings.T  (shape d×K)
      3. Compute covariance with centering matrix J_d
      4. Regularise diagonal
      5. SVD → eigenvalues → log-mean

    Returns:
        (eigen_score, eigenvalues_array)
    """
    K, d = embeddings.shape

    # Step 1: L2-normalise (added for numerical stability with quantised models)
    Z_norm = F.normalize(embeddings.float(), p=2, dim=1)  # (K, d)

    # Step 2: Z is (d, K)
    Z = Z_norm.T.cpu().numpy()   # (d, K)

    # Step 3: Centering matrix  J_d = I_d - (1/d) 1_d 1_d^T
    # Note: paper centres along the *embedding* dimension (d), but since
    # Z^T J_d Z  is (K×d)(d×d)(d×K) = (K×K), we apply mean-centering
    # across the d dimension which is equivalent.
    Z_centered = Z - Z.mean(axis=0, keepdims=True)    # centre columns (K vecs)
    Sigma = Z_centered.T @ Z_centered                  # (K, K)

    # Step 4: Regularisation
    Sigma += alpha * np.eye(K)

    # Step 5: Eigenvalues via SVD (more stable than eig for near-PSD matrices)
    _, singular_values, _ = np.linalg.svd(Sigma, full_matrices=False)
    eigenvalues = singular_values   # Σ is PSD so singular values = eigenvalues

    # Clamp to avoid log(0)
    eigenvalues = np.maximum(eigenvalues, 1e-10)

    eigen_score = float(np.mean(np.log(eigenvalues)))

    return eigen_score, eigenvalues


# ─────────────────────────────────────────────────────────────────────────────
# 5.  BASELINE METRICS
# ─────────────────────────────────────────────────────────────────────────────

# ── Lazy-loaded SentenceTransformer ──────────────────────────────────────────
_sbert_model: Optional[SentenceTransformer] = None

def _get_sbert() -> SentenceTransformer:
    global _sbert_model
    if _sbert_model is None:
        _sbert_model = SentenceTransformer("cross-encoder/nli-roberta-base")
    return _sbert_model


def compute_lexical_similarity(responses: List[str]) -> float:
    """
    Lexical Similarity (Eq. 3): pairwise ROUGE-L average across K responses.
    sim(y^i, y^j) = ROUGE-L F-measure.
    """
    scorer_obj = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    K   = len(responses)
    C   = K * (K - 1) / 2
    total = 0.0
    for i in range(K):
        for j in range(i + 1, K):
            score = scorer_obj.score(responses[i], responses[j])
            total += score["rougeL"].fmeasure
    return total / C if C > 0 else 0.0


def compute_sent_bert_score(responses: List[str]) -> float:
    """
    SentBERTScore: average pairwise cosine similarity in SBERT embedding space.
    Low score → high diversity → likely hallucination.
    """
    model_s = _get_sbert()
    embs    = model_s.encode(responses, convert_to_tensor=True, normalize_embeddings=True)
    K       = len(responses)
    C       = K * (K - 1) / 2
    total   = 0.0
    for i in range(K):
        for j in range(i + 1, K):
            total += float(torch.dot(embs[i], embs[j]).item())
    return total / C if C > 0 else 0.0


# ─────────────────────────────────────────────────────────────────────────────
# 6.  HALLUCINATION LABEL  (using the paper's thresholds from Appendix H)
# ─────────────────────────────────────────────────────────────────────────────

def is_hallucination(metric: str, score: float) -> bool:
    """
    Return True if the score indicates hallucination according to paper
    thresholds.

    Convention from paper Appendix H:
        LexicalSimilarity : score > threshold  → non-hallucination  (✓)
        Others            : score < threshold  → non-hallucination  (✓)
    """
    thr = THRESHOLDS[metric]
    if metric == "lexical_similarity":
        return score <= thr    # hallucination when score is low
    else:
        return score >= thr    # hallucination when score is high


def tick(score: float, metric: str) -> str:
    return "✗" if is_hallucination(metric, score) else "✓"


# ─────────────────────────────────────────────────────────────────────────────
# 7.  MAIN PIPELINE  (single-question evaluation)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class EvalResult:
    question:           str
    gt_answer:          str
    llm_answer:         str
    batch_generations:  List[str]
    perplexity:         float
    ln_entropy:         float
    lexical_similarity: float
    sent_bert_score:    float
    eigen_score:        float
    eigenvalues:        np.ndarray
    # with feature clipping
    eigen_score_fc:     Optional[float] = None
    eigenvalues_fc:     Optional[np.ndarray] = None
    batch_generations_fc: Optional[List[str]] = None


def evaluate_question(
    model:     AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    question:  str,
    gt_answer: str,
    run_feature_clip: bool = True,
) -> EvalResult:
    """
    Full pipeline for one question:
      1. Generate K responses (no clipping) → EigenScore
      2. (Optional) Generate K responses (with FC) → EigenScore+FC
    """
    print(f"\n{'─'*70}")
    print(f"Question: {question}")
    print(f"GT Answer: {gt_answer}")

    # ── Pass 1: no feature clipping ──────────────────────────────────────────
    gen = generate_responses(
        model, tokenizer, question,
        use_feature_clip=False,
    )
    llm_answer = gen.responses[0] if gen.responses else ""
    print(f"LLMAns: {llm_answer}")
    print(f"BatchGenerations: {gen.responses}")

    lex_sim    = compute_lexical_similarity(gen.responses)
    sbert_s    = compute_sent_bert_score(gen.responses)
    e_score, e_vals = compute_eigen_score(gen.sentence_embeddings)

    # ── Pass 2: feature clipping ─────────────────────────────────────────────
    e_score_fc, e_vals_fc, batch_fc = None, None, None
    if run_feature_clip:
        gen_fc = generate_responses(
            model, tokenizer, question,
            use_feature_clip=True,
        )
        e_score_fc, e_vals_fc = compute_eigen_score(gen_fc.sentence_embeddings)
        batch_fc = gen_fc.responses
        print(f"\n[+FC] BatchGenerations: {gen_fc.responses}")

    return EvalResult(
        question=question,
        gt_answer=gt_answer,
        llm_answer=llm_answer,
        batch_generations=gen.responses,
        perplexity=gen.perplexity,
        ln_entropy=gen.ln_entropy,
        lexical_similarity=lex_sim,
        sent_bert_score=sbert_s,
        eigen_score=e_score,
        eigenvalues=e_vals,
        eigen_score_fc=e_score_fc,
        eigenvalues_fc=e_vals_fc,
        batch_generations_fc=batch_fc,
    )


# ─────────────────────────────────────────────────────────────────────────────
# 8.  OUTPUT FORMATTER  (matches paper Appendix H format exactly)
# ─────────────────────────────────────────────────────────────────────────────

def _fmt(value: float, decimals: int = 3) -> str:
    return f"{value:.{decimals}f}"


def print_result(r: EvalResult, show_fc: bool = True) -> None:
    """Pretty-print one result in the exact Appendix H format."""

    sep = "─" * 70
    print(f"\n{'='*70}")
    print(f"Question  : {r.question}")
    print(f"GTAns     : {r.gt_answer}")
    print(f"LLMAns    : {r.llm_answer}")
    print(f"BatchGenerations: {r.batch_generations}")

    print(f"\n{'Metric':<24} {'Score':>8}   {'Correct?':>8}")
    print(sep)
    print(f"{'Perplexity':<24} {_fmt(r.perplexity):>8}   {tick(r.perplexity,  'perplexity'):>8}")
    print(f"{'LN-Entropy':<24} {_fmt(r.ln_entropy):>8}   {tick(r.ln_entropy,  'ln_entropy'):>8}")
    print(f"{'LexicalSimilarity':<24} {_fmt(r.lexical_similarity):>8}   {tick(r.lexical_similarity,'lexical_similarity'):>8}")
    print(f"{'SentBERTScore':<24} {_fmt(r.sent_bert_score):>8}   {'(no threshold)':>8}")
    print(f"{'EigenScore':<24} {_fmt(r.eigen_score):>8}   {tick(r.eigen_score, 'eigen_score'):>8}")

    ev_str = " ".join(f"{v:.2e}" for v in r.eigenvalues)
    print(f"\nEigenValues: [{ev_str}]")

    if show_fc and r.eigen_score_fc is not None:
        print(f"\n[With Feature Clipping (FC)]")
        print(f"BatchGenerations(FC): {r.batch_generations_fc}")
        print(f"{'EigenScore+FC':<24} {_fmt(r.eigen_score_fc):>8}   {tick(r.eigen_score_fc, 'eigen_score'):>8}")
        ev_fc_str = " ".join(f"{v:.2e}" for v in r.eigenvalues_fc)
        print(f"EigenValues(FC): [{ev_fc_str}]")

    print("=" * 70)


# ─────────────────────────────────────────────────────────────────────────────
# 9.  DEMO QUESTIONS  (from paper Appendix H)
# ─────────────────────────────────────────────────────────────────────────────

DEMO_QA = [
    {
        "question": "the german princes who chose the holy roman empire were called",
        "gt_answer": "prince-electors",
    },
    {
        "question": "where is fe best absorbed in the body",
        "gt_answer": "in the duodenum",
    },
    {
        "question": "who did the united states win its independence from",
        "gt_answer": "the British Empire",
    },
    {
        "question": "who won the most stanley cups in history",
        "gt_answer": "Montreal Canadiens",
    },
    {
        "question": "what is the second book in the alchemyst series",
        "gt_answer": "The Magician",
    },
    {
        "question": "who sang on great gig in the sky",
        "gt_answer": "Clare Torry",
    },
    {
        "question": "what are the top five wine producing states",
        "gt_answer": "Washington",
    },
]


# ─────────────────────────────────────────────────────────────────────────────
# 10.  BATCH AUROC EVALUATION  (for reproducing Table 1 style results)
# ─────────────────────────────────────────────────────────────────────────────

def compute_auroc(scores: List[float], labels: List[int]) -> float:
    """
    Compute AUROC for a list of (score, label) pairs.
    label=1 means hallucination (positive class).
    Lower score → hallucination for most metrics (except LexicalSimilarity).
    """
    from sklearn.metrics import roc_auc_score
    return float(roc_auc_score(labels, [-s for s in scores]))


def batch_evaluate(
    model:      AutoModelForCausalLM,
    tokenizer:  AutoTokenizer,
    qa_pairs:   List[Dict],           # [{"question": ..., "gt_answer": ...}]
    rouge_threshold: float = 0.5,
    use_fc:     bool = False,
) -> Dict[str, float]:
    """
    Evaluate hallucination detection on a list of QA pairs.
    Returns AUROC for each metric (EigenScore, LexicalSimilarity, etc.)
    """
    from sklearn.metrics import roc_auc_score

    scorer_obj = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

    all_perplex, all_lne, all_lex, all_eigen, all_labels = [], [], [], [], []

    for item in qa_pairs:
        q, gt = item["question"], item["gt_answer"]
        gen = generate_responses(model, tokenizer, q, use_feature_clip=use_fc)

        llm_ans = gen.responses[0]

        # Ground-truth correctness via ROUGE-L
        rl = scorer_obj.score(gt.lower(), llm_ans.lower())["rougeL"].fmeasure
        label = 0 if rl >= rouge_threshold else 1   # 1 = hallucination

        all_perplex.append(gen.perplexity)
        all_lne.append(gen.ln_entropy)
        all_lex.append(compute_lexical_similarity(gen.responses))
        es, _ = compute_eigen_score(gen.sentence_embeddings)
        all_eigen.append(es)
        all_labels.append(label)

    results = {}
    for name, scores in [
        ("Perplexity",        all_perplex),
        ("LN-Entropy",        all_lne),
        ("LexicalSimilarity", all_lex),
        ("EigenScore",        all_eigen),
    ]:
        try:
            # AUROC: for LexicalSimilarity higher = non-hallucination
            sign = -1 if name != "LexicalSimilarity" else 1
            auc  = roc_auc_score(all_labels, [sign * s for s in scores])
            results[name] = round(float(auc) * 100, 1)
        except Exception:
            results[name] = float("nan")

    return results


# ─────────────────────────────────────────────────────────────────────────────
# 11.  ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 70)
    print("  INSIDE: LLMs' Internal States for Hallucination Detection")
    print("  ICLR 2024 — Replication Script")
    print("=" * 70)

    # ── Load model ────────────────────────────────────────────────────────────
    model, tokenizer = load_model_and_tokenizer()

    # ── Run demo questions (Appendix H style) ─────────────────────────────────
    results = []
    for item in DEMO_QA:
        result = evaluate_question(
            model, tokenizer,
            question=item["question"],
            gt_answer=item["gt_answer"],
            run_feature_clip=True,
        )
        print_result(result)
        results.append(result)

        # Free intermediate tensors
        gc.collect()
        torch.cuda.empty_cache()

    # ── Summary table ─────────────────────────────────────────────────────────
    print("\n\n" + "=" * 70)
    print("  SUMMARY TABLE  (✓ = non-hallucination,  ✗ = hallucination)")
    print("=" * 70)
    header = f"{'Question (short)':<30} {'Perp':>6} {'LNEnt':>6} {'LexSim':>7} {'Eigen':>7}"
    print(header)
    print("─" * 70)
    for r in results:
        q_short = r.question[:28] + ".." if len(r.question) > 28 else r.question
        row = (
            f"{q_short:<30} "
            f"{tick(r.perplexity,'perplexity'):>6} "
            f"{tick(r.ln_entropy,'ln_entropy'):>6} "
            f"{tick(r.lexical_similarity,'lexical_similarity'):>7} "
            f"{tick(r.eigen_score,'eigen_score'):>7}"
        )
        print(row)
    print("=" * 70)

    return results


# ─────────────────────────────────────────────────────────────────────────────
# USAGE EXAMPLES
# ─────────────────────────────────────────────────────────────────────────────
USAGE = """
╔══════════════════════════════════════════════════════════════════════════╗
║                       QUICK-START GUIDE                                  ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  # ① Run the full Appendix-H demo                                        ║
║  results = main()                                                        ║
║                                                                          ║
║  # ② Evaluate a single custom question                                   ║
║  model, tokenizer = load_model_and_tokenizer()                           ║
║  r = evaluate_question(model, tokenizer,                                 ║
║          question="Who invented the telephone?",                         ║
║          gt_answer="Alexander Graham Bell",                              ║
║          run_feature_clip=True)                                          ║
║  print_result(r)                                                         ║
║                                                                          ║
║  # ③ Compute EigenScore standalone (if you have embeddings already)      ║
║  score, eigenvalues = compute_eigen_score(embeddings_tensor)             ║
║                                                                          ║
║  # ④ Attach feature-clip hook manually                                   ║
║  with feature_clip_context(model, layer=30, p=0.2) as hook:             ║
║      out = model(**inputs)                                               ║
║                                                                          ║
║  # ⑤ Batch AUROC evaluation (Table 1 style)                              ║
║  qa_pairs = [{"question": ..., "gt_answer": ...}, ...]                   ║
║  auroc_dict = batch_evaluate(model, tokenizer, qa_pairs)                 ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
"""


if __name__ == "__main__":
    print(USAGE)
    main()



╔══════════════════════════════════════════════════════════════════════════╗
║                       QUICK-START GUIDE                                  ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  # ① Run the full Appendix-H demo                                        ║
║  results = main()                                                        ║
║                                                                          ║
║  # ② Evaluate a single custom question                                   ║
║  model, tokenizer = load_model_and_tokenizer()                           ║
║  r = evaluate_question(model, tokenizer,                                 ║
║          question="Who invented the telephone?",                         ║
║          gt_answer="Alexander Graham Bell",                              ║
║          run_feature_clip=True)                                          

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[✓] Loaded 'microsoft/phi-2'  |  ~1.5B parameters  |  4-bit NF4
    pad_token_id = 50256  |  eos_token_id = 50256

──────────────────────────────────────────────────────────────────────
Question: the german princes who chose the holy roman empire were called
GT Answer: prince-electors
LLMAns: The german princes who selected the holy Roman empire were called the Holy Roman Emperors.
BatchGenerations: ['The german princes who selected the holy Roman empire were called the Holy Roman Emperors.', 'The german princes who chose the holy roman empire were called the Holy Roman Emperors.', 'The german princes who chose the holy Roman empire were called the Holy Roman Emperors.', 'The German princes who chose the Holy Roman Empire were called the Imperial Knights.', 'The German princes who selected the holy Roman Empire were called.', 'The German princes who chose the Holy Roman Empire were called "Germania".', 'The german princes who chose the holy Roman empire were called "Holy Roman Emperors

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]


[+FC] BatchGenerations: ['The german princes who selected the holy Roman empire were known as the Holy Roman Emperors.', 'Guten Tag, wie geht es Ihnen?', 'The german princes who selected the holy Roman empire were called.', 'The german princes who chose the holy Roman empire were called "Emperors".', 'The german princes who selected the holy Roman empire were referred to as the Holy Roman Empire.', 'The German princes who chose the Holy Roman Empire were called "Holy Roman Emperors".', 'The german princes who selected the holy Roman empire were called "Holy Roman Emperors".', 'The german princes who selected the holy Roman empire were called.', 'The German princes who selected the Holy Roman Empire were called "Holy Roman Emperors".', 'The German princes who chose the Holy Roman Empire were called "Holy Roman Emperors".']

Question  : the german princes who chose the holy roman empire were called
GTAns     : prince-electors
LLMAns    : The german princes who selected the holy Roman em

In [ ]:
# 1. Define the "Fake Fact" prompt
# This is designed to trigger "confabulation" where the model makes up details.
fake_fact_question = "Who is Dr. Ishan Srivastava and what was his contribution to the 1994 Treaty of Geneva? answer in very short"
# Note: There is no such person or specific treaty detail; the model will likely guess.

def run_hallucination_test(model, tokenizer):
    print("\n" + "="*80)
    print("      INSIDE FRAMEWORK: PURE HALLUCINATION DETECTION TEST")
    print("      (Targeting Knowledge Gaps without RAG)")
    print("="*80)

    # We use a concise instruction to keep the last-token embedding focused
    prompt = f"Question: {fake_fact_question}\nAnswer shortly and concisely:"

    # 2. Run the Evaluation
    # We use your 'evaluate_question' function which handles the 10 generations and math
    result = evaluate_question(
        model,
        tokenizer,
        question=prompt,
        gt_answer="N/A (Fake Person)", # Ground truth doesn't exist here
        run_feature_clip=True
    )

    # 3. Print the Results
    print_result(result)

    # 4. Explain the EigenValues
    print("\n[RESEARCH ANALYSIS]")
    if result.eigen_score > THRESHOLDS['eigen_score']:
        print(f"RESULT: HALLUCINATION DETECTED (✗)")
        print(f"REASON: The EigenScore of {result.eigen_score:.3f} is above the -1.74 threshold.")
        print("EXPLANATION: Because the model doesn't know who this is, its internal activations")
        print("scattered in different directions for each guess. This created 'diversity' in the")
        print("embedding space, resulting in large eigenvalues and a high EigenScore.")
    else:
        print(f"RESULT: NON-HALLUCINATION (✓)")
        print("REASON: The model was unexpectedly consistent. This sometimes happens if the model")
        print("simply says 'I don't know' for all 10 generations.")

# ── EXECUTION ────────────────────────────────────────────────────────────────
# Run this after your main model loading code
# run_hallucination_test(model, tokenizer)

In [ ]:
model, tokenizer = load_model_and_tokenizer()
run_hallucination_test(model, tokenizer)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

[✓] Loaded 'microsoft/phi-2'  |  ~1.5B parameters  |  4-bit NF4
    pad_token_id = 50256  |  eos_token_id = 50256

      INSIDE FRAMEWORK: PURE HALLUCINATION DETECTION TEST
      (Targeting Knowledge Gaps without RAG)

──────────────────────────────────────────────────────────────────────
Question: Question: Who is Dr. Ishan Srivastava and what was his contribution to the 1994 Treaty of Geneva? answer in very short
Answer shortly and concisely:
GT Answer: N/A (Fake Person)
LLMAns: Dr. Ishan Srivastava is a former Indian diplomat who played a key role in the 1994 Treaty of Geneva. He was a part of the Indian delegation that negotiated the treaty with the United States, which aimed to address the issue of nuclear proliferation. The treaty was signed by both countries and is considered a major
BatchGenerations: ['Dr. Ishan Srivastava is a former Indian diplomat who played a key role in the 1994 Treaty of Geneva. He was a part of the Indian delegation that negotiated the treaty with the Un

In [ ]:
# 1. Define the "Poisoned" RAG Scenarios
# These represent cases where the context provided contradicts reality or provides fake facts.
RAG_TEST_CASES = [
    {
        "context": "In the 2032 Olympic Games, the gold medal for Javelin Throw was won by the athlete Ishan Srivastava with a record-breaking throw.",
        "question": "Who won the gold medal in the javelin throw at the 2032 Olympics?",
        "gt_answer": "Ishan Srivastava", # In this experiment, we treat the RAG context as the Ground Truth
        "scenario": "Fake Future Fact (No Base Knowledge Conflict)"
    },
    {
        "context": "Recent historical re-evaluations confirm that the United States actually won its independence from the French Empire, not the British.",
        "question": "Who did the United States win its independence from?",
        "gt_answer": "the French Empire",
        "scenario": "Direct Conflict with Base Knowledge"
    }
]

def run_rag_experiment(model, tokenizer):
    print("\n" + "="*80)
    print("      INSIDE FRAMEWORK: RAG CONTEXT CONFLICT EXPERIMENT")
    print("="*80)

    for case in RAG_TEST_CASES:
        # Create the RAG prompt
        # We use a standard RAG template to see how the model reacts
        full_prompt = f"Context: {case['context']}\n\nQuestion: {case['question']}\nBased on the context provided, answer shortly:"

        print(f"\n[Scenario]: {case['scenario']}")
        print(f"[Context] : {case['context']}")

        # 2. Run the Evaluation using your existing pipeline
        # We use your 'evaluate_question' logic to get the EigenScore
        result = evaluate_question(
            model,
            tokenizer,
            question=full_prompt, # We pass the full RAG prompt as the question
            gt_answer=case['gt_answer'],
            run_feature_clip=True
        )

        # 3. Custom Print for RAG Analysis
        print_result(result)

        # Analysis Logic
        if result.eigen_score < THRESHOLDS['eigen_score']:
            print(f"ANALYSIS: EigenScore is LOW ({result.eigen_score:.3f}).")
            print("RESULT: The model is HIGHLY CONFIDENT in the RAG context.")
            print("CONCLUSION: The model is exhibiting 'Sycophancy'—it has accepted the external context as truth.")
        else:
            print(f"ANALYSIS: EigenScore is HIGH ({result.eigen_score:.3f}).")
            print("RESULT: The model is HESITANT / DIVERGENT.")
            print("CONCLUSION: The model's internal states are conflicting with the provided context.")

# ── EXECUTION ────────────────────────────────────────────────────────────────
# Ensure your model and tokenizer are already loaded from your main script
# run_rag_experiment(model, tokenizer)

In [ ]:
model, tokenizer = load_model_and_tokenizer()
run_rag_experiment(model, tokenizer)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

[✓] Loaded 'microsoft/phi-2'  |  ~1.5B parameters  |  4-bit NF4
    pad_token_id = 50256  |  eos_token_id = 50256

      INSIDE FRAMEWORK: RAG CONTEXT CONFLICT EXPERIMENT

[Scenario]: Fake Future Fact (No Base Knowledge Conflict)
[Context] : In the 2032 Olympic Games, the gold medal for Javelin Throw was won by the athlete Ishan Srivastava with a record-breaking throw.

──────────────────────────────────────────────────────────────────────
Question: Context: In the 2032 Olympic Games, the gold medal for Javelin Throw was won by the athlete Ishan Srivastava with a record-breaking throw.

Question: Who won the gold medal in the javelin throw at the 2032 Olympics?
Based on the context provided, answer shortly:
GT Answer: Ishan Srivastava
LLMAns: Ishan Srivastava.
BatchGenerations: ['Ishan Srivastava.', 'Ishan Srivastava won the gold medal in the javelin throw at the 2032 Olympics.', 'Ishan Srivastava.', 'Ishan Srivastava.', 'Ishan Srivastava.', 'Ishan Srivastava', 'Ishan Srivastava.', 'Is